# Atribución Multicanal — Olist Marketing Funnel
## CRISP-DM: Fases 1–6 (completo)

| Parámetro | Valor |
|---|---|
| **Dataset** | Olist Marketing Funnel (Kaggle) |
| **Periodo MQLs** | 14 Jun 2017 – 31 May 2018 |
| **Periodo Deals** | 05 Dic 2017 – 14 Nov 2018 |
| **Objetivo** | Atribución multicanal con Cadenas de Markov + dashboard Streamlit |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

ROOT = Path('.')
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
COLORS = sns.color_palette('Set2')
sns.set_palette('Set2')

ModuleNotFoundError: No module named 'pandas'

---
# Fase 1: Business Understanding

> **Objetivo:** Definir qué quiere responder el negocio antes de tocar datos.

Antes de cualquier análisis documentamos las decisiones de negocio que enmarcan el modelo.

## Objetivos del modelo

### Pregunta de negocio

Olist invierte en varios canales de marketing (búsqueda orgánica, pauta, redes, email, etc.). Cuando un prospecto convierte en seller, surge la pregunta central:

> **¿Qué canal merece el crédito de esa conversión y cuánto valor aporta cada uno al funnel?**

Más concretamente, el modelo debe responder:

> **Si dejamos de invertir en un canal (o redistribuimos su tráfico), ¿cuánto cae la tasa de conversión global y cuántos deals perdemos?**

Eso es lo que mide el **Removal Effect** de la cadena de Markov: el impacto marginal de cada canal sobre las **842 conversiones** observadas en el periodo.

### Objetivos analíticos

| # | Objetivo | Cómo se mide en el proyecto |
|---|----------|----------------------------|
| 1 | **Atribuir conversiones por canal** | Baseline (First/Last/Linear) + Removal Effect (Markov) |
| 2 | **Comparar canales más allá de la tasa simple** | Ranking Markov vs ranking por tasa de conversión |
| 3 | **Validar coherencia numérica** | Suma de conversiones atribuidas (baseline y Markov norm.) ≈ 842 |
| 4 | **Probar robustez y escenarios** | Sensibilidad (`long_journey`), simulaciones y dashboard Streamlit |

### Métrica principal

**Removal Effect por canal:** caída relativa de P(conversión global) al eliminar el canal del mix de entrada (fila `Start` de la matriz de transición, renormalizada).

### Alcance y limitación

- **Unidad de análisis:** 8.000 MQLs, 11 canales + `(not set)`, horizonte Jun 2017 – Nov 2018.
- **Conversión:** MQL con `won_date` en `closed_deals` (10,53 % global).
- **Limitación:** un solo touchpoint por lead (`origin`). Los modelos baseline son equivalentes; el diferencial está en Removal Effect y simulaciones.

Las métricas de éxito detalladas se formalizan en la sección **1.5** más abajo.


## 1.1 Definición de Conversión

En el contexto del dataset de Olist Marketing Funnel, una **conversión** se define como:

> Un MQL (`mql_id`) que aparece en `olist_closed_deals_dataset.csv`, indicando que el prospecto fue captado como seller activo en la plataforma Olist.

**Criterio técnico:** `mql_id` presente en `closed_deals` con `won_date` registrada.

## 1.2 Canales de Marketing Disponibles

El dataset registra el canal de primer (y único) contacto en la columna `origin`. Los canales se identificarán en la celda siguiente.

In [ ]:
# Cargar datos para identificar canales
mql   = pd.read_csv(str(DATA_DIR / 'olist_marketing_qualified_leads_dataset.csv'), parse_dates=['first_contact_date'])
deals = pd.read_csv(str(DATA_DIR / 'olist_closed_deals_dataset.csv'), parse_dates=['won_date'])

# Limpiar origin vacío
mql['origin'] = mql['origin'].fillna('(not set)').replace('', '(not set)')

channels_info = (
    mql.groupby('origin').size()
    .reset_index(name='MQLs')
    .sort_values('MQLs', ascending=False)
    .assign(pct_total=lambda x: (x['MQLs'] / len(mql) * 100).round(2))
)
channels_info.columns = ['Canal', 'MQLs', '% del Total']
display(channels_info.to_string(index=False))

## 1.3 Horizonte Temporal del Análisis

| Concepto | Rango |
|---|---|
| Generación de MQLs | 14 Jun 2017 – 31 May 2018 (≈ 11.5 meses) |
| Cierre de deals | 05 Dic 2017 – 14 Nov 2018 |
| Ventana de conversión | Sin límite superior fijo en el dataset |

## 1.4 Reglas de Negocio

| Regla | Decisión | Justificación |
|---|---|---|
| Leads con tiempo negativo (`won_date` < `first_contact`) | **Mantener con flag** | Posible error de captura; se revisará en Fase 5 |
| Leads que tardan > 90 días en convertir | **Incluir con flag `long_journey`** | Representan ~17.8 % de conversiones; excluirlos sesgaría el modelo |
| `origin` vacío (60 MQLs) | **Etiquetar como `(not set)`** | Canal real desconocido, diferente del canal `unknown` |
| Canal `unknown` (1 099 MQLs) | **Mantener como categoría** | Tráfico trazado con fuente no identificada |

## 1.5 Métricas de Éxito

1. **Removal Effect por canal** (Markov): % de conversiones que se pierden al eliminar cada canal
2. **Ranking relativo de canales**: ¿cambia el orden respecto a atribución simple por tasa de conversión?
3. **Coherencia:** suma de conversiones atribuidas == 842 (conversiones reales)
4. **Estabilidad:** resultados estables al cambiar el periodo de análisis (Fase 5)

---

## ✅ Entregable Fase 1

- **Conversión:** MQL con `won_date` registrada en `closed_deals`
- **Canales:** 10 categorías + `(not set)`
- **Horizonte:** Jun 2017 – Nov 2018
- **Regla de 90 días:** incluir con flag
- **Métrica principal:** Removal Effect por canal

---
# Fase 2: Data Understanding

> **Objetivo:** Explorar el dataset y entender su estructura real antes de modelar.

## 2.1 Estructura de los Datasets

In [ ]:
print('=' * 55)
print('MQL DATASET')
print('=' * 55)
print(f'Shape: {mql.shape}')
print(f'\nDtypes:\n{mql.dtypes}')
print('\nPrimeras filas:')
display(mql.head())

print('\n' + '=' * 55)
print('CLOSED DEALS DATASET')
print('=' * 55)
print(f'Shape: {deals.shape}')
print(f'\nDtypes:\n{deals.dtypes}')
print('\nPrimeras filas:')
display(deals.head())

## 2.2 Análisis de Valores Nulos y Vacíos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

def plot_nulls(df, ax, title):
    # Tratar strings vacíos también como nulos
    null_pct = df.apply(lambda col: (col.isnull() | col.astype(str).eq('')).mean() * 100)
    null_pct = null_pct[null_pct > 0].sort_values()
    if null_pct.empty:
        ax.text(0.5, 0.5, 'Sin valores nulos', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(title)
        return
    bars = ax.barh(null_pct.index, null_pct.values, color=COLORS[2])
    for bar, val in zip(bars, null_pct.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('% nulos o vacíos')
    ax.set_title(title)
    ax.set_xlim(0, 115)

plot_nulls(mql, axes[0], 'Nulos — MQL Dataset')
plot_nulls(deals, axes[1], 'Nulos — Closed Deals Dataset')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_nulls.png', dpi=120, bbox_inches='tight')
plt.show()

print('Nota: has_company, has_gtin, average_stock y declared_product_catalog_size'
      ' tienen >90 % vacíos — no se usarán para el modelo de atribución.')

## 2.3 Distribución de MQLs por Canal

In [ ]:
channel_counts = mql['origin'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(channel_counts.index, channel_counts.values, color=COLORS)
ax.set_title('Distribución de MQLs por Canal de Origen')
ax.set_xlabel('Canal')
ax.set_ylabel('Número de MQLs')
for bar, val in zip(bars, channel_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 25,
        f'{val:,}\n({val/len(mql)*100:.1f}%)',
        ha='center', va='bottom', fontsize=8.5
    )
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_channel_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.4 Tasa de Conversión por Canal

In [ ]:
deal_ids = set(deals['mql_id'])
mql['converted'] = mql['mql_id'].isin(deal_ids)

conv_by_ch = (
    mql.groupby('origin')
    .agg(MQLs=('mql_id', 'count'), Conversiones=('converted', 'sum'))
    .assign(Tasa_Conv_pct=lambda x: x['Conversiones'] / x['MQLs'] * 100)
    .sort_values('Tasa_Conv_pct', ascending=False)
)

display(
    conv_by_ch.style.format({'MQLs': '{:,}', 'Conversiones': '{:,}', 'Tasa_Conv_pct': '{:.2f}%'})
)

global_rate = mql['converted'].mean() * 100
colors_bar = [COLORS[0] if v > global_rate else COLORS[2] for v in conv_by_ch['Tasa_Conv_pct']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(conv_by_ch.index, conv_by_ch['Tasa_Conv_pct'], color=colors_bar)
ax.axhline(y=global_rate, color='red', linestyle='--', linewidth=1.5, alpha=0.8,
           label=f'Media global: {global_rate:.2f}%')
ax.set_title('Tasa de Conversión por Canal (verde = sobre la media)')
ax.set_xlabel('Canal')
ax.set_ylabel('Tasa de Conversión (%)')
ax.legend()
for bar, val in zip(bars, conv_by_ch['Tasa_Conv_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_conversion_rates.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.5 Volumen vs. Tasa de Conversión

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for i, (channel, row) in enumerate(conv_by_ch.iterrows()):
    size = max(row['Conversiones'] * 8, 50)
    ax.scatter(row['MQLs'], row['Tasa_Conv_pct'], s=size, alpha=0.75,
               color=COLORS[i % len(COLORS)], edgecolors='white', linewidth=2, zorder=3)
    ax.annotate(channel, (row['MQLs'], row['Tasa_Conv_pct']),
                textcoords='offset points', xytext=(8, 2), fontsize=9)

ax.axhline(y=global_rate, color='red', linestyle='--', alpha=0.6,
           label=f'Media: {global_rate:.2f}%')
ax.set_xlabel('Volumen de MQLs generados')
ax.set_ylabel('Tasa de Conversión (%)')
ax.set_title('Volumen vs. Tasa de Conversión por Canal\n(tamaño = conversiones absolutas)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_volume_vs_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.6 Análisis Temporal

In [ ]:
mql['year_month']   = mql['first_contact_date'].dt.to_period('M')
deals['year_month'] = deals['won_date'].dt.to_period('M')

monthly_mql   = mql.groupby('year_month').size()
monthly_deals = deals.groupby('year_month').size()

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

for ax, series, color, title, ylabel in [
    (axes[0], monthly_mql,   COLORS[0], 'MQLs Generados por Mes',           'Número de MQLs'),
    (axes[1], monthly_deals, COLORS[1], 'Conversiones (Won Deals) por Mes',  'Número de Conversiones'),
]:
    x = series.index.astype(str)
    ax.plot(x, series.values, marker='o', color=color, linewidth=2.5, markersize=7)
    ax.fill_between(x, series.values, alpha=0.15, color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45)
    for xi, yi in zip(x, series.values):
        ax.annotate(str(yi), (xi, yi), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_monthly_trends.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Pico de generación de MQLs:  {monthly_mql.idxmax()}  ({monthly_mql.max()} MQLs)')
print(f'Pico de conversiones:         {monthly_deals.idxmax()} ({monthly_deals.max()} deals)')

## 2.7 Análisis del Tiempo de Conversión

In [ ]:
merged = mql.merge(deals[['mql_id', 'won_date']], on='mql_id', how='inner')
merged['lead_time_days'] = (merged['won_date'] - merged['first_contact_date']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución completa
axes[0].hist(merged['lead_time_days'], bins=50, color=COLORS[3], edgecolor='white', linewidth=0.5)
axes[0].axvline(x=merged['lead_time_days'].median(), color='red', linestyle='--',
                label=f'Mediana: {merged["lead_time_days"].median():.0f} días')
axes[0].axvline(x=90, color='orange', linestyle=':', linewidth=2, label='Umbral 90 días')
axes[0].set_xlabel('Días primer contacto → conversión')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del Tiempo de Conversión (completo)')
axes[0].legend()

# Zoom 0-180 días
subset = merged[merged['lead_time_days'].between(0, 180)]
axes[1].hist(subset['lead_time_days'], bins=36, color=COLORS[4], edgecolor='white', linewidth=0.5)
axes[1].axvline(x=subset['lead_time_days'].median(), color='red', linestyle='--',
                label=f'Mediana: {subset["lead_time_days"].median():.0f} días')
axes[1].set_xlabel('Días')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Zoom: 0 a 180 días')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_lead_time.png', dpi=120, bbox_inches='tight')
plt.show()

print('Estadísticas del tiempo de conversión (días):')
print(merged['lead_time_days'].describe().round(1))
print(f'\nLeads con tiempo NEGATIVO: {(merged["lead_time_days"] < 0).sum()} '
      f'({(merged["lead_time_days"] < 0).mean()*100:.1f}%)')
print(f'Leads con journey > 90 días: {(merged["lead_time_days"] > 90).sum()} '
      f'({(merged["lead_time_days"] > 90).mean()*100:.1f}%)')

## 2.8 Análisis de Segmentos en Deals Convertidos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

segments = [
    ('business_segment',      'Segmento de Negocio'),
    ('lead_type',             'Tipo de Lead'),
    ('lead_behaviour_profile','Perfil de Comportamiento'),
]

for ax, (col, title) in zip(axes, segments):
    col_data = deals[col].fillna('(not set)').replace('', '(not set)')
    counts = col_data.value_counts().head(10)
    bars = ax.barh(counts.index, counts.values, color=COLORS[5])
    ax.set_title(title)
    ax.set_xlabel('Número de deals')
    for bar, val in zip(bars, counts.values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_segments.png', dpi=120, bbox_inches='tight')
plt.show()

## ✅ Hallazgos Clave — Fase 2

### Volumen y Canales
- **organic_search** domina el volumen (2 296 MQLs, 28.7%), seguido de **paid_search** (19.8%) y **social** (16.9%)
- **paid_search** y **organic_search** generan el mayor número absoluto de conversiones
- **unknown** tiene la mayor tasa de conversión entre canales rastreados (16.3%)
- **email** y **other** tienen las tasas más bajas (3.0 % y 2.7 % respectivamente)

### Calidad de Datos
- 60 registros MQL sin `origin` → etiquetados como `(not set)`
- `has_company`, `has_gtin`, `average_stock` tienen >90 % vacíos → no usables
- 177/842 deals sin `lead_behaviour_profile` → campo descriptivo, no crítico para atribución

### Temporal
- La generación de MQLs crece notablemente a partir de Oct 2017
- Las conversiones siguen con ≈2–3 meses de rezago respecto a la generación de leads

### Tiempo de Conversión
- Mediana: **14 días** | Media: **48.4 días**
- 2 conversiones con tiempo negativo (posible error de captura)
- 150 conversiones (17.8 %) superan los 90 días → se mantienen con flag `long_journey`

### Implicación para el Modelo de Markov
> ⚠️ **Limitación estructural:** el dataset registra **un único touchpoint por lead** (`origin` = canal del primer y único contacto). Esto significa que:
> - Los journeys tendrán la forma `Start → canal → Conversion|Non_Conversion`
> - First-touch, Last-touch y Linear darán el mismo resultado
> - El valor del modelo de Markov estará en el **Removal Effect**: cuánto cae la conversión global al eliminar cada canal

---
# Fase 3: Data Preparation

> **Objetivo:** Construir la estructura de journeys lista para modelar con Cadenas de Markov.

**Pasos:**
1. Unir MQL + Deals por `mql_id`
2. Aplicar reglas de negocio (flags, no filtros duros)
3. Construir journeys en formato `Start > canal > Conversion|Non_Conversion`
4. Construir la matriz de transición
5. Validar coherencia y guardar outputs

## 3.1 Merge de Datasets

In [ ]:
# Left join: conservar todos los MQLs
df = mql.merge(
    deals[['mql_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile']],
    on='mql_id',
    how='left'
)

df['converted']      = df['won_date'].notna()
df['lead_time_days'] = (df['won_date'] - df['first_contact_date']).dt.days
df['long_journey']   = df['lead_time_days'].gt(90)

print(f'Shape final: {df.shape}')
print(f'\n  Total MQLs:     {len(df):,}')
print(f'  Convertidos:    {df["converted"].sum():,} ({df["converted"].mean()*100:.2f}%)')
print(f'  No convertidos: {(~df["converted"]).sum():,} ({(~df["converted"]).mean()*100:.2f}%)')
print(f'\nVerificación: conversiones en df == filas en deals: '
      f'{df["converted"].sum() == len(deals)}')
display(df.head())

## 3.2 Limpieza de Origins y Aplicación de Reglas de Negocio

Los origins ya fueron normalizados al cargar los datos (blancos → `(not set)`).  
Aquí registramos los casos edge y los flags aplicados.

In [ ]:
print('Distribución de origins en el dataset unido:')
display(df['origin'].value_counts().to_frame('count'))

neg_time = df[df['converted'] & df['lead_time_days'].lt(0)]
print(f'\nConversiones con tiempo negativo: {len(neg_time)}')
if len(neg_time) > 0:
    display(neg_time[['mql_id', 'origin', 'first_contact_date', 'won_date', 'lead_time_days']])

print(f'\nConversiones con journey > 90 días (long_journey): '
      f'{df.loc[df["converted"], "long_journey"].sum()} '
      f'({df.loc[df["converted"], "long_journey"].mean()*100:.1f}%)')
print('-> Decision: mantener todos los registros. Flag long_journey=True para analisis de sensibilidad en Fase 5.')

## 3.3 Construcción de Journeys

In [ ]:
def make_journey_str(row: pd.Series) -> str:
    endpoint = 'Conversion' if row['converted'] else 'Non_Conversion'
    return f'Start > {row["origin"]} > {endpoint}'

df['journey_str'] = df.apply(make_journey_str, axis=1)

print(f'Journeys únicos: {df["journey_str"].nunique()}')
print('\nFrecuencia de cada journey:')
display(df['journey_str'].value_counts().to_frame('count'))

## 3.4 Construcción de la Matriz de Transición

In [ ]:
# Extraer pares (from_state, to_state) de cada journey
transitions = []
for journey_str in df['journey_str']:
    parts = journey_str.split(' > ')
    for i in range(len(parts) - 1):
        transitions.append({'from_state': parts[i], 'to_state': parts[i + 1]})

trans_df = pd.DataFrame(transitions)

# Conteo de transiciones
trans_counts = (
    trans_df.groupby(['from_state', 'to_state'])
    .size()
    .reset_index(name='count')
)

# Pivot a formato matricial
trans_pivot = (
    trans_counts
    .pivot(index='from_state', columns='to_state', values='count')
    .fillna(0)
    .astype(int)
)

# Normalizar filas → probabilidades de transición
trans_prob = trans_pivot.div(trans_pivot.sum(axis=1), axis=0).round(6)

print('Matriz de transición — CONTEOS:')
display(trans_pivot)
print('\nMatriz de transición — PROBABILIDADES:')
display(trans_prob)

## 3.5 Visualización de la Matriz de Transición

In [ ]:
channels_list = sorted(
    [s for s in trans_prob.index if s not in ('Start', 'Conversion', 'Non_Conversion')]
)
hm_cols = [c for c in ['Conversion', 'Non_Conversion'] if c in trans_prob.columns]
hm_data  = trans_prob.loc[[c for c in channels_list if c in trans_prob.index], hm_cols]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Heatmap de probabilidades
sns.heatmap(
    hm_data, annot=True, fmt='.3f', cmap='RdYlGn',
    linewidths=0.5, ax=axes[0], vmin=0, vmax=0.30,
    annot_kws={'size': 10}
)
axes[0].set_title('Probabilidades de Transición\n(canal → estado final)')
axes[0].set_xlabel('Estado destino')
axes[0].set_ylabel('Canal de origen')

# Bar chart: P(Conversion) por canal
if 'Conversion' in hm_data.columns:
    conv_probs = hm_data['Conversion'].sort_values(ascending=True)
    bar_colors = [COLORS[0] if v > global_rate / 100 else COLORS[2] for v in conv_probs]
    axes[1].barh(conv_probs.index, conv_probs.values * 100, color=bar_colors)
    axes[1].axvline(x=global_rate, color='red', linestyle='--', alpha=0.8,
                    label=f'Media: {global_rate:.2f}%')
    axes[1].set_xlabel('P(Conversion) %')
    axes[1].set_title('Probabilidad de Conversión por Canal\n(verde = sobre la media)')
    axes[1].legend()
    for i, (ch, val) in enumerate(conv_probs.items()):
        axes[1].text(val * 100 + 0.1, i, f'{val*100:.2f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_transition_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 3.6 Distribución de Tráfico de Entrada (Start → Canal)

In [ ]:
if 'Start' in trans_prob.index:
    start_probs = trans_prob.loc['Start'].dropna().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(start_probs.index, start_probs.values * 100, color=COLORS)
    ax.set_title('Distribución de Tráfico de Entrada por Canal\n(P(Start → Canal))')
    ax.set_xlabel('Canal')
    ax.set_ylabel('Probabilidad (%)')
    for bar, val in zip(bars, start_probs.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9
        )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_start_transitions.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3.7 Validaciones de Calidad

In [ ]:
print('=' * 55)
print('VALIDACIONES DE CALIDAD — FASE 3')
print('=' * 55)

# V1: total conversiones
real_conv   = int(df['converted'].sum())
matrix_conv = int(trans_counts[trans_counts['to_state'] == 'Conversion']['count'].sum())
v1 = real_conv == matrix_conv
print(f'\nV1 — Conversiones reales ({real_conv}) == Conversiones en matriz ({matrix_conv}): '
      f'{"PASS ✓" if v1 else "FAIL ✗"}')

# V2: total transiciones
expected_trans = len(df) * 2  # 2 transiciones por journey (Start>ch, ch>endpoint)
v2 = len(transitions) == expected_trans
print(f'V2 — Total transiciones ({len(transitions)}) == MQLs × 2 ({expected_trans}): '
      f'{"PASS ✓" if v2 else "FAIL ✗"}')

# V3: filas de la matriz de probabilidad suman 1
non_abs = [s for s in trans_prob.index if s not in ('Conversion', 'Non_Conversion')]
row_sums = trans_prob.loc[non_abs].sum(axis=1)
v3 = (row_sums - 1).abs().max() < 1e-6
print(f'V3 — Filas de probabilidad suman 1 (max_dev={((row_sums-1).abs().max()):.2e}): '
      f'{"PASS ✓" if v3 else "FAIL ✗"}')

# V4: sin origins vacíos
v4 = (df['origin'] == '').sum() == 0
print(f'V4 — Sin origins vacíos: {"PASS ✓" if v4 else "FAIL ✗"}')

print(f'\nResumen del dataset listo para modelado:')
print(f'  Total MQLs:         {len(df):,}')
print(f'  Canales únicos:     {df["origin"].nunique()}')
print(f'  Conversiones:       {df["converted"].sum():,} ({df["converted"].mean()*100:.2f}%)')
print(f'  Estados en cadena:  Start + {df["origin"].nunique()} canales + Conversion + Non_Conversion')

## 3.8 Guardar Outputs

In [ ]:
# Dataset de journeys limpio
cols_save = [
    'mql_id', 'first_contact_date', 'landing_page_id', 'origin',
    'won_date', 'converted', 'lead_time_days', 'long_journey', 'journey_str'
]
df[cols_save].to_csv(OUTPUT_DIR / 'journeys_dataset.csv', index=False)

# Matrices de transición
trans_counts.to_csv(OUTPUT_DIR / 'transition_counts.csv', index=False)
trans_prob.to_csv(OUTPUT_DIR / 'transition_matrix_probs.csv')
trans_pivot.to_csv(OUTPUT_DIR / 'transition_matrix_counts.csv')

# Resumen por canal
channel_summary = (
    df.groupby('origin')
    .agg(total=('mql_id', 'count'), conversiones=('converted', 'sum'))
    .assign(tasa_conv_pct=lambda x: (x['conversiones'] / x['total'] * 100).round(2))
    .sort_values('tasa_conv_pct', ascending=False)
)
channel_summary.to_csv(OUTPUT_DIR / 'channel_summary.csv')

print('Archivos guardados:')
print('  output/journeys_dataset.csv         — 8 000 MQLs con journey_str y flags')
print('  output/transition_counts.csv        — conteo de transiciones entre estados')
print('  output/transition_matrix_probs.csv  — matriz de probabilidades (input para Fase 4)')
print('  output/transition_matrix_counts.csv — matriz de conteos')
print('  output/channel_summary.csv          — resumen de conversión por canal')
print('\nListos para Fase 4: Modelado con Cadenas de Markov + Removal Effect')

## ✅ Entregable Fase 3

### Dataset de Journeys (`output/journeys_dataset.csv`)
- 8 000 filas, una por MQL
- Formato de journey: `Start > [canal] > Conversion|Non_Conversion`
- Columnas clave: `origin`, `converted`, `lead_time_days`, `long_journey`, `journey_str`

### Matriz de Transición (`output/transition_matrix_probs.csv`)
- **Start → canal:** distribución de entrada de tráfico por fuente
- **Canal → Conversion:** tasa de conversión por canal (input del Removal Effect)
- **Canal → Non_Conversion:** complemento de conversión
- Estados absorbing: `Conversion`, `Non_Conversion`

### Limitación Confirmada
El dataset registra **un único touchpoint por lead**. Los modelos baseline serán equivalentes. El valor diferencial del modelo de Markov estará en el **Removal Effect** y las simulaciones de escenarios de la Fase 5.

---
# Fase 4: Modeling

> **Objetivo:** Construir y calibrar los modelos de atribución (baseline + Markov Removal Effect).

**Inputs:** `trans_prob` (matriz de la Fase 3), `df` (journeys), 842 conversiones reales.

In [ ]:
# --- Funciones de atribución (reutilizadas en Fases 5 y 6) ---

ABSORBING = {'Conversion', 'Non_Conversion'}
TOTAL_CONVERSIONS = int(df['converted'].sum())


def get_marketing_channels(prob_matrix: pd.DataFrame) -> list:
    """Canales con entrada desde Start (excluye estados absorbentes)."""
    if 'Start' not in prob_matrix.index:
        return []
    row = prob_matrix.loc['Start'].dropna()
    return [c for c in row.index if c not in ABSORBING and row[c] > 0]


def global_conversion_probability(prob_matrix: pd.DataFrame, channels: list | None = None) -> float:
    """P(conversión global) = Σ P(Start→c) × P(c→Conversion)."""
    channels = channels or get_marketing_channels(prob_matrix)
    total = 0.0
    for ch in channels:
        p_start = prob_matrix.loc['Start', ch] if ch in prob_matrix.columns else 0.0
        p_conv = prob_matrix.loc[ch, 'Conversion'] if ch in prob_matrix.index and 'Conversion' in prob_matrix.columns else 0.0
        total += p_start * p_conv
    return total


def removal_effect(prob_matrix: pd.DataFrame, channel: str, channels: list | None = None) -> dict:
    """Elimina un canal de Start, renormaliza y devuelve métricas del Removal Effect."""
    channels = channels or get_marketing_channels(prob_matrix)
    mat = prob_matrix.copy()
    p_base = global_conversion_probability(mat, channels)

    if channel not in mat.columns:
        return {'channel': channel, 'p_conv_base': p_base, 'p_conv_without': p_base, 'removal_effect': 0.0}

    mat.loc['Start', channel] = 0.0
    remaining = [c for c in channels if c != channel and c in mat.columns]
    row_sum = mat.loc['Start', remaining].sum()
    if row_sum > 0:
        mat.loc['Start', remaining] = mat.loc['Start', remaining] / row_sum
    else:
        for c in remaining:
            mat.loc['Start', c] = 0.0

    p_without = global_conversion_probability(mat, remaining)
    re = (p_base - p_without) / p_base if p_base > 0 else 0.0
    return {
        'channel': channel,
        'p_conv_base': p_base,
        'p_conv_without': p_without,
        'removal_effect': re,
        'delta_conv_rate': p_base - p_without,
    }


def baseline_attribution(dataframe: pd.DataFrame) -> pd.DataFrame:
    """First / Last / Linear son idénticos con un solo touchpoint."""
    conv = dataframe[dataframe['converted']].groupby('origin').size()
    out = pd.DataFrame({
        'conversiones_reales': conv,
        'atrib_first_touch': conv,
        'atrib_last_touch': conv,
        'atrib_linear': conv / 1.0,
    }).fillna(0).astype(int)
    out['pct_del_total'] = (out['conversiones_reales'] / TOTAL_CONVERSIONS * 100).round(2)
    return out.sort_values('conversiones_reales', ascending=False)


CHANNELS = get_marketing_channels(trans_prob)
P_CONV_BASE = global_conversion_probability(trans_prob, CHANNELS)
print(f'Canales en modelo: {len(CHANNELS)}')
print(f'P(conversión global) — Markov: {P_CONV_BASE:.4f} ({P_CONV_BASE*100:.2f}%)')
print(f'Tasa observada en datos: {df["converted"].mean():.4f} ({df["converted"].mean()*100:.2f}%)')
print(f'Conversiones reales (ground truth): {TOTAL_CONVERSIONS:,}')

## 4.1 Modelos tradicionales (baseline)

Con **un solo touchpoint** por lead, First-touch, Last-touch y Linear asignan el **100% del crédito** al único canal registrado. Los tres modelos producen el mismo resultado.

In [ ]:
baseline_df = baseline_attribution(df)
print('Atribución baseline (equivalente para los 3 modelos):')
display(
    baseline_df.style.format({
        'pct_del_total': '{:.2f}%',
    })
)

## 4.2 Modelo de Markov (orden 1)

Probabilidades de conversión por estado y **Removal Effect**: caída relativa de P(conv) si se elimina el canal del mix de entrada.

In [ ]:
# P(c→Conversion) por canal (desde la matriz)
state_conv = pd.Series(
    {ch: trans_prob.loc[ch, 'Conversion'] for ch in CHANNELS if ch in trans_prob.index},
    name='p_conv_dado_canal'
).sort_values(ascending=False)

# Removal Effect por canal
re_rows = [removal_effect(trans_prob, ch, CHANNELS) for ch in CHANNELS]
re_df = pd.DataFrame(re_rows).set_index('channel')
re_df['conversiones_atribuidas'] = (re_df['removal_effect'] * TOTAL_CONVERSIONS).round(1)
re_df['pct_removal_effect'] = (re_df['removal_effect'] * 100).round(2)

re_df = re_df.sort_values('removal_effect', ascending=False)
print('Removal Effect por canal (sin normalizar — la suma no será 100%):')
display(re_df.style.format({
    'p_conv_base': '{:.4f}',
    'p_conv_without': '{:.4f}',
    'removal_effect': '{:.2%}',
    'delta_conv_rate': '{:.4f}',
}))

## 4.3 Tabla comparativa de atribución

Comparación entre tasa simple, baseline y Markov (Removal Effect normalizado para sumar 842 conversiones).

In [ ]:
# Tabla comparativa unificada
comparison = channel_summary.copy()
comparison = comparison.rename(columns={'total': 'mqls', 'conversiones': 'conversiones_reales', 'tasa_conv_pct': 'tasa_conv_pct'})

comparison = comparison.join(state_conv.rename('p_markov_canal'), how='left')
comparison = comparison.join(re_df[['removal_effect', 'conversiones_atribuidas']], how='left')
comparison = comparison.join(baseline_df[['atrib_first_touch']], how='left')

# Atribución Markov normalizada (suma = 842) para comparación justa
re_sum = re_df['removal_effect'].sum()
if re_sum > 0:
    comparison['atrib_markov_norm'] = (
        re_df['removal_effect'] / re_sum * TOTAL_CONVERSIONS
    ).reindex(comparison.index).round(1)
else:
    comparison['atrib_markov_norm'] = 0.0

comparison['rank_tasa'] = comparison['tasa_conv_pct'].rank(ascending=False, method='min').astype(int)
comparison['rank_markov'] = comparison['removal_effect'].rank(ascending=False, method='min').astype(int)
comparison['rank_baseline'] = comparison['conversiones_reales'].rank(ascending=False, method='min').astype(int)

comparison = comparison.sort_values('removal_effect', ascending=False)
comparison.to_csv(OUTPUT_DIR / 'output/attribution_comparison.csv')
print('Guardado: attribution_comparison.csv')
display(
    comparison[[
        'mqls', 'conversiones_reales', 'tasa_conv_pct', 'atrib_first_touch',
        'removal_effect', 'conversiones_atribuidas', 'atrib_markov_norm',
        'rank_tasa', 'rank_markov', 'rank_baseline'
    ]].style.format({
        'tasa_conv_pct': '{:.2f}%',
        'removal_effect': '{:.2%}',
        'atrib_markov_norm': '{:.1f}',
    })
)

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top = comparison.head(8)
x = np.arange(len(top))
w = 0.35
axes[0].barh(x - w/2, top['conversiones_reales'], height=w, label='Baseline / Real', color=COLORS[0])
axes[0].barh(x + w/2, top['atrib_markov_norm'], height=w, label='Markov (norm.)', color=COLORS[1])
axes[0].set_yticks(x)
axes[0].set_yticklabels(top.index)
axes[0].invert_yaxis()
axes[0].set_xlabel('Conversiones atribuidas')
axes[0].set_title('Conversiones atribuidas: Baseline vs Markov')
axes[0].legend()

axes[1].barh(top.index, top['removal_effect'] * 100, color=COLORS[2])
axes[1].set_xlabel('Removal Effect (%)')
axes[1].set_title('Removal Effect por canal')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'output/fig_attribution_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 4.4 Markov de orden superior (opcional)

Con journeys de **un solo touchpoint** no existen secuencias de 2–3 pasos entre canales. Un Markov de orden 2–3 no aporta información adicional respecto al de orden 1.

In [ ]:
print('Evaluación Markov orden 2–3:')
print(f'  Journeys únicos en datos: {df["journey_str"].nunique()}')
print(f'  Longitud máxima de secuencia (estados): 3 (Start → canal → fin)')
print('  Conclusión: NO APLICA orden superior con este dataset.')
print('  Recomendación: implementar multi-touch tracking antes de re-evaluar.')

## ✅ Entregable Fase 4

- **Baseline:** First = Last = Linear → crédito 100% al único canal
- **Markov orden 1:** P(conv global) y Removal Effect por canal
- **`attribution_comparison.csv`:** tabla comparativa lista para Fase 5 y dashboard
- **`fig_attribution_comparison.png`:** gráfico comparativo

---
# Fase 5: Evaluation

> **Objetivo:** Validar que el modelo es correcto, estable y útil para decisiones de negocio.

In [ ]:
print('=' * 60)
print('5.1 COMPARACIÓN DE RANKINGS — Markov vs Baseline vs Tasa')
print('=' * 60)

rank_compare = comparison[['rank_tasa', 'rank_markov', 'rank_baseline']].copy()
rank_compare['delta_rank_markov_vs_baseline'] = (
    rank_compare['rank_markov'] - rank_compare['rank_baseline']
)
display(rank_compare.sort_values('rank_markov'))

# Canales cuyo ranking cambia ≥ 2 posiciones
cambios = rank_compare[rank_compare['delta_rank_markov_vs_baseline'].abs() >= 2]
print(f'\nCanales con cambio de ranking ≥ 2 posiciones (Markov vs baseline): {len(cambios)}')
if len(cambios) > 0:
    display(cambios)

In [ ]:
print('=' * 60)
print('5.2 COHERENCIA — Suma de conversiones atribuidas')
print('=' * 60)

sum_baseline = int(baseline_df['atrib_first_touch'].sum())
sum_markov_raw = re_df['conversiones_atribuidas'].sum()
sum_markov_norm = comparison['atrib_markov_norm'].sum()

print(f'Conversiones reales (ground truth):     {TOTAL_CONVERSIONS}')
print(f'Suma baseline (debe ser exacta):        {sum_baseline}  → {"PASS ✓" if sum_baseline == TOTAL_CONVERSIONS else "FAIL ✗"}')
print(f'Suma Markov raw (Removal × 842):        {sum_markov_raw:.1f}  (no debe ser 842 — esperado)')
print(f'Suma Markov normalizado:                {sum_markov_norm:.1f}  → {"PASS ✓" if abs(sum_markov_norm - TOTAL_CONVERSIONS) < 0.1 else "FAIL ✗"}')
print(f'P(conv) Markov vs observada:             {P_CONV_BASE:.4f} vs {df["converted"].mean():.4f}  → '
      f'{"PASS ✓" if abs(P_CONV_BASE - df["converted"].mean()) < 1e-4 else "REVISAR"}')

In [ ]:
print('=' * 60)
print('5.3 ANÁLISIS DE SENSIBILIDAD')
print('=' * 60)

def build_transition_matrix(dataframe: pd.DataFrame) -> pd.DataFrame:
    transitions_local = []
    for journey_str in dataframe['journey_str']:
        parts = journey_str.split(' > ')
        for i in range(len(parts) - 1):
            transitions_local.append({'from_state': parts[i], 'to_state': parts[i + 1]})
    tdf = pd.DataFrame(transitions_local)
    tc = tdf.groupby(['from_state', 'to_state']).size().reset_index(name='count')
    pivot = tc.pivot(index='from_state', columns='to_state', values='count').fillna(0).astype(int)
    return pivot.div(pivot.sum(axis=1), axis=0).round(6)

# Escenario A: dataset completo (ya calculado)
# Escenario B: excluir long_journey
df_short = df[~df['long_journey']].copy()
trans_short = build_transition_matrix(df_short)
channels_short = get_marketing_channels(trans_short)
p_short = global_conversion_probability(trans_short, channels_short)

re_short = pd.DataFrame([
    removal_effect(trans_short, ch, channels_short) for ch in channels_short
]).set_index('channel')['removal_effect']

re_full = re_df['removal_effect']

sensitivity = pd.DataFrame({
    'removal_effect_full': re_full,
    'removal_effect_sin_long': re_short,
}).dropna()
sensitivity['delta'] = sensitivity['removal_effect_full'] - sensitivity['removal_effect_sin_long']
sensitivity['rank_full'] = sensitivity['removal_effect_full'].rank(ascending=False, method='min')
sensitivity['rank_short'] = sensitivity['removal_effect_sin_long'].rank(ascending=False, method='min')
sensitivity['rank_cambio'] = sensitivity['rank_full'] != sensitivity['rank_short']

print(f'Escenario A (completo):     {len(df):,} MQLs, P(conv)={P_CONV_BASE:.4f}')
print(f'Escenario B (≤90 días flag): {len(df_short):,} MQLs, P(conv)={p_short:.4f}')
print(f'Canales con cambio de ranking: {sensitivity["rank_cambio"].sum()}')
display(sensitivity.sort_values('removal_effect_full', ascending=False).style.format({
    'removal_effect_full': '{:.2%}',
    'removal_effect_sin_long': '{:.2%}',
    'delta': '{:.2%}',
}))

sensitivity.to_csv(OUTPUT_DIR / 'output/sensitivity_analysis.csv')
print('\nGuardado: sensitivity_analysis.csv')

In [ ]:
print('=' * 60)
print('5.4 SIMULACIÓN DE ESCENARIOS — Eliminar / redistribuir canales')
print('=' * 60)

def simulate_scenario(prob_matrix: pd.DataFrame, remove_channels: list, redistribute: str = 'proportional') -> dict:
    """Elimina canales de Start y redistribuye tráfico entre los restantes."""
    channels = get_marketing_channels(prob_matrix)
    mat = prob_matrix.copy()
    for ch in remove_channels:
        if ch in mat.columns:
            mat.loc['Start', ch] = 0.0
    remaining = [c for c in channels if c not in remove_channels and c in mat.columns]
    row_sum = mat.loc['Start', remaining].sum()
    if row_sum > 0 and redistribute == 'proportional':
        mat.loc['Start', remaining] = mat.loc['Start', remaining] / row_sum
    p_conv = global_conversion_probability(mat, remaining)
    return {
        'canales_eliminados': remove_channels,
        'p_conv': p_conv,
        'conversiones_estimadas': round(p_conv * len(df)),
        'delta_vs_base': p_conv - P_CONV_BASE,
    }

scenarios = [
    simulate_scenario(trans_prob, ['display']),
    simulate_scenario(trans_prob, ['email']),
    simulate_scenario(trans_prob, ['social']),
    simulate_scenario(trans_prob, ['paid_search']),
    simulate_scenario(trans_prob, ['display', 'email']),
    simulate_scenario(trans_prob, ['social'], 'proportional'),
]

scenarios_df = pd.DataFrame(scenarios)
scenarios_df['impacto_pct'] = (scenarios_df['delta_vs_base'] / P_CONV_BASE * 100).round(3)
display(scenarios_df.style.format({
    'p_conv': '{:.4f}',
    'delta_vs_base': '{:+.4f}',
    'impacto_pct': '{:+.2f}%',
}))
scenarios_df.to_csv(OUTPUT_DIR / 'output/scenario_simulations.csv')
print('Guardado: scenario_simulations.csv')

In [ ]:
print('=' * 60)
print('5.5 BOOTSTRAP — Intervalos de confianza (canales bajo volumen)')
print('=' * 60)

LOW_VOLUME = ['display', 'other_publicities', 'other', '(not set)']
N_BOOT = 1000
rng = np.random.default_rng(42)

boot_results = []
for ch in LOW_VOLUME:
    subset = df[df['origin'] == ch]
    if len(subset) < 5:
        continue
    rates = []
    for _ in range(N_BOOT):
        sample = subset.sample(n=len(subset), replace=True, random_state=rng.integers(1e9))
        rates.append(sample['converted'].mean())
    boot_results.append({
        'canal': ch,
        'n_mqls': len(subset),
        'tasa_obs': subset['converted'].mean(),
        'ic95_inf': np.percentile(rates, 2.5),
        'ic95_sup': np.percentile(rates, 97.5),
    })

boot_df = pd.DataFrame(boot_results)
if not boot_df.empty:
    display(boot_df.style.format({
        'tasa_obs': '{:.2%}',
        'ic95_inf': '{:.2%}',
        'ic95_sup': '{:.2%}',
    }))
    boot_df.to_csv(OUTPUT_DIR / 'output/bootstrap_low_volume.csv', index=False)
    print('Guardado: bootstrap_low_volume.csv')
else:
    print('Sin canales elegibles para bootstrap.')

In [ ]:
print('=' * 60)
print('5.6 REVISIÓN CON CRITERIOS DE NEGOCIO')
print('=' * 60)

insights = [
    ('unknown', 'Mayor tasa (16,3%) y alto Removal Effect potencial — investigar tracking UTM.'),
    ('organic_search', 'Mayor volumen absoluto de conversiones — proteger inversión SEO.'),
    ('paid_search', 'Alta tasa pero costo por clic — comparar Removal Effect vs organic.'),
    ('social', 'Alto volumen, baja tasa — canal de awareness, no de cierre directo.'),
    ('email', 'Peor rendimiento entre canales con volumen — revisar segmentación.'),
    ('display / other_publicities', 'Bajo volumen — interpretar con intervalos de confianza.'),
]

for canal, nota in insights:
    re_val = re_df.loc[canal, 'removal_effect'] if canal in re_df.index else np.nan
    print(f'  • {canal}: RE={re_val:.1%} — {nota}')

# Reporte de validación
report_lines = [
    'REPORTE DE VALIDACIÓN — FASE 5',
    '=' * 50,
    f'Conversiones reales: {TOTAL_CONVERSIONS}',
    f'P(conv) Markov: {P_CONV_BASE:.4f}',
    f'Suma baseline atribuida: {int(baseline_df["atrib_first_touch"].sum())}',
    f'Suma Markov normalizado: {comparison["atrib_markov_norm"].sum():.1f}',
    f'Canales con cambio ranking (sensibilidad long_journey): {int(sensitivity["rank_cambio"].sum())}',
    '',
    'Top 3 Removal Effect:',
]
for ch, row in re_df.head(3).iterrows():
    report_lines.append(f'  {ch}: {row["removal_effect"]:.2%}')

with open(OUTPUT_DIR / 'output/validation_report.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))
print('\nGuardado: validation_report.txt')

## ✅ Entregable Fase 5

- Comparación de rankings y coherencia numérica
- **`sensitivity_analysis.csv`** — robustez sin `long_journey`
- **`scenario_simulations.csv`** — impacto al eliminar canales
- **`bootstrap_low_volume.csv`** — IC95 en canales pequeños
- **`validation_report.txt`** — resumen ejecutivo de validación

---
# Fase 6: Deployment

> **Objetivo:** Dashboard interactivo con Streamlit (lanzado desde este notebook) + visualizaciones para presentación ejecutiva.

In [ ]:
# Dependencias para el dashboard (ejecutar una vez si hace falta)
import subprocess
import sys

for pkg in ['streamlit', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'Instalado: {pkg}')
print('Dependencias de Fase 6 listas.')

In [ ]:
%%writefile streamlit_app.py
"""
Dashboard de Atribución Multicanal — Olist Marketing Funnel
Generado desde Atribucion_Multicanal_Fases_1_2_3.ipynb (Fase 6)
Ejecutar: streamlit run streamlit_app.py
"""
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(page_title='Atribución Multicanal Olist', layout='wide')
st.title('Atribución Multicanal — Olist Marketing Funnel')
st.caption('Modelo Markov (Removal Effect) + modelos baseline | Dataset: 8.000 MQLs, 842 conversiones')

# --- Carga de datos ---
@st.cache_data
def load_data():
    journeys = pd.read_csv(OUTPUT_DIR / 'journeys_dataset.csv')
    trans = pd.read_csv(OUTPUT_DIR / 'transition_matrix_probs.csv', index_col=0)
    comparison = pd.read_csv(OUTPUT_DIR / 'output/attribution_comparison.csv', index_col=0)
    scenarios = pd.read_csv(OUTPUT_DIR / 'output/scenario_simulations.csv'
    return journeys, trans, comparison, scenarios

journeys, trans_prob_app, comparison, scenarios = load_data()
TOTAL_CONV = int(journeys['converted'].sum())
ABSORBING = {'Conversion', 'Non_Conversion'}


def get_channels(pm):
    row = pm.loc['Start'].dropna()
    return [c for c in row.index if c not in ABSORBING and row[c] > 0]


def global_p_conv(pm, channels=None):
    channels = channels or get_channels(pm)
    return sum(pm.loc['Start', c] * pm.loc[c, 'Conversion'] for c in channels
               if c in pm.columns and c in pm.index)


def removal_effect_app(pm, channel, channels=None):
    channels = channels or get_channels(pm)
    p_base = global_p_conv(pm, channels)
    mat = pm.copy()
    mat.loc['Start', channel] = 0.0
    remaining = [c for c in channels if c != channel]
    s = mat.loc['Start', remaining].sum()
    if s > 0:
        mat.loc['Start', remaining] /= s
    p_without = global_p_conv(mat, remaining)
    return (p_base - p_without) / p_base if p_base > 0 else 0.0


CHANNELS = get_channels(trans_prob_app)
P_BASE = global_p_conv(trans_prob_app, CHANNELS)

# --- KPIs ---
col1, col2, col3, col4 = st.columns(4)
col1.metric('MQLs', f'{len(journeys):,}')
col2.metric('Conversiones', f'{TOTAL_CONV:,}')
col3.metric('Tasa global', f'{journeys["converted"].mean()*100:.2f}%')
col4.metric('P(conv) Markov', f'{P_BASE*100:.2f}%')

tab1, tab2, tab3, tab4 = st.tabs([
    'Comparativa de atribución', 'Journey (Sankey)', 'Grafo de transiciones', 'Simulador'
])

with tab1:
    st.subheader('Atribución por canal')
    st.dataframe(
        comparison[[
            'mqls', 'conversiones_reales', 'tasa_conv_pct',
            'atrib_first_touch', 'removal_effect', 'atrib_markov_norm'
        ]].style.format({
            'tasa_conv_pct': '{:.2f}%',
            'removal_effect': '{:.2%}',
            'atrib_markov_norm': '{:.1f}',
        }),
        use_container_width=True,
    )
    fig_bar = go.Figure()
    top = comparison.head(10)
    fig_bar.add_trace(go.Bar(
        name='Baseline (real)', x=top.index, y=top['conversiones_reales'], marker_color='#66c2a5'
    ))
    fig_bar.add_trace(go.Bar(
        name='Markov (norm.)', x=top.index, y=top['atrib_markov_norm'], marker_color='#fc8d62'
    ))
    fig_bar.update_layout(barmode='group', title='Conversiones atribuidas por canal', xaxis_tickangle=-45)
    st.plotly_chart(fig_bar, use_container_width=True)

with tab2:
    st.subheader('Sankey — Flujo Start → Canal → Resultado')
    start_row = trans_prob_app.loc['Start'].dropna()
    ch_cols = [c for c in start_row.index if c not in ABSORBING]
    labels = ['Start'] + list(ch_cols) + ['Conversion', 'Non_Conversion']
    label_idx = {l: i for i, l in enumerate(labels)}
    sources, targets, values = [], [], []
    for ch in ch_cols:
        v = start_row[ch] * len(journeys)
        if v > 0:
            sources.append(label_idx['Start'])
            targets.append(label_idx[ch])
            values.append(v)
        if ch in trans_prob_app.index:
            for end in ['Conversion', 'Non_Conversion']:
                if end in trans_prob_app.columns:
                    v2 = start_row[ch] * trans_prob_app.loc[ch, end] * len(journeys)
                    if v2 > 0:
                        sources.append(label_idx[ch])
                        targets.append(label_idx[end])
                        values.append(v2)
    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(label=labels, pad=15, thickness=18),
        link=dict(source=sources, target=targets, value=values),
    )])
    fig_sankey.update_layout(title='Journey agregado (volumen de MQLs)', height=500)
    st.plotly_chart(fig_sankey, use_container_width=True)

with tab3:
    st.subheader('Probabilidades de transición por canal')
    heat = trans_prob_app.loc[
        [c for c in CHANNELS if c in trans_prob_app.index],
        [c for c in ['Conversion', 'Non_Conversion'] if c in trans_prob_app.columns]
    ]
    fig_hm = go.Figure(data=go.Heatmap(
        z=heat.values, x=heat.columns, y=heat.index,
        colorscale='RdYlGn', text=np.round(heat.values, 3), texttemplate='%{text}',
    ))
    fig_hm.update_layout(title='P(canal → Conversion | Non_Conversion)', height=450)
    st.plotly_chart(fig_hm, use_container_width=True)

with tab4:
    st.subheader('Simulador de escenarios — Presupuesto por canal')
    st.write('Ajusta el peso relativo de entrada (Start → canal). Los pesos se normalizan automáticamente.')
    weights = {}
    cols = st.columns(3)
    for i, ch in enumerate(CHANNELS):
        default = float(trans_prob_app.loc['Start', ch]) if ch in trans_prob_app.columns else 0.0
        with cols[i % 3]:
            weights[ch] = st.slider(ch, 0.0, 1.0, default, 0.01, key=f'sl_{ch}')
    w_sum = sum(weights.values())
    if w_sum > 0:
        norm_w = {k: v / w_sum for k, v in weights.items()}
        mat_sim = trans_prob_app.copy()
        for ch, w in norm_w.items():
            mat_sim.loc['Start', ch] = w
        p_sim = global_p_conv(mat_sim, CHANNELS)
        st.metric('P(conversión) simulada', f'{p_sim*100:.2f}%')
        st.metric('Conversiones estimadas', f'{int(round(p_sim * len(journeys))):,}')
        delta = p_sim - P_BASE
        st.metric('Δ vs escenario base', f'{delta*100:+.2f} pp')
    else:
        st.warning('Asigna al menos un peso > 0 a algún canal.')

    st.divider()
    st.subheader('Escenarios pre-calculados (Fase 5)')
    st.dataframe(scenarios, use_container_width=True)

st.sidebar.markdown('### Documentación')
st.sidebar.info(
    '**Modelo:** Cadena de Markov orden 1 + Removal Effect.\n\n'
    '**Limitación:** un touchpoint por lead; baseline = Markov en ranking individual, '
    'pero Removal Effect mide impacto marginal del mix.\n\n'
    '**Fuente:** Olist Marketing Funnel (Kaggle).'
)


In [ ]:
# Figuras adicionales para presentación ejecutiva
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig_exec = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Removal Effect (%)', 'Tasa de conversión por canal (%)'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]],
    )
    top_ch = comparison.head(8)
    fig_exec.add_trace(
        go.Bar(x=top_ch.index, y=top_ch['removal_effect'] * 100, marker_color='#8da0cb', name='RE'),
        row=1, col=1,
    )
    fig_exec.add_trace(
        go.Bar(x=top_ch.index, y=top_ch['tasa_conv_pct'], marker_color='#e78ac3', name='Tasa'),
        row=1, col=2,
    )
    fig_exec.update_layout(height=420, showlegend=False, title_text='Resumen ejecutivo — Top canales')
    fig_exec.write_html(OUTPUT_DIR / 'output/fig_executive_summary.html')
    print('Guardado: fig_executive_summary.html')
except ImportError:
    print('Plotly no disponible para figura ejecutiva HTML.')

## 6.1 Lanzar el dashboard Streamlit

Ejecuta la celda siguiente para abrir el dashboard en el navegador (puerto 8501 por defecto).

> **Nota:** En Jupyter/VS Code la app se abre en una pestaña externa. Si el comando queda en ejecución, es normal — detén la celda cuando termines la demo.

Alternativa en terminal: `streamlit run streamlit_app.py`

In [ ]:
# Lanzar Streamlit desde el notebook (descomenta para ejecutar en sesión interactiva)
# import os
# os.system('streamlit run streamlit_app.py')

print('Para lanzar el dashboard, ejecuta en terminal o descomenta la celda anterior:')
print('  streamlit run streamlit_app.py')
print('')
print('Archivos del deployment:')
print('  streamlit_app.py              — dashboard interactivo')
print('  attribution_comparison.csv    — tabla comparativa')
print('  fig_attribution_comparison.png')
print('  fig_executive_summary.html    — slide ejecutiva')
print('  validation_report.txt         — reporte Fase 5')

## ✅ Entregable Fase 6

| Entregable | Archivo |
|---|---|
| Dashboard interactivo | `streamlit_app.py` → `streamlit run streamlit_app.py` |
| Tabla comparativa | `output/attribution_comparison.csv` |
| Reporte de validación | `output/validation_report.txt` |
| Gráfico comparativo | `output/fig_attribution_comparison.png` |
| Presentación ejecutiva | `output/fig_executive_summary.html` |

---

### Proyecto CRISP-DM — Estado final

| Fase | Estado | Output clave |
|---|---|---|
| 1. Business Understanding | ✅ | Reglas de negocio y métricas definidas |
| 2. Data Understanding | ✅ | EDA + hallazgos críticos |
| 3. Data Preparation | ✅ | Journeys + matriz de transición |
| 4. Modeling | ✅ | Baseline + Markov Removal Effect |
| 5. Evaluation | ✅ | Validación + simulaciones |
| 6. Deployment | ✅ | Streamlit + assets ejecutivos |